In [42]:
import faultier
import serial
import time

ft = faultier.Faultier()
ser = serial.Serial(ft.get_serial_path(), baudrate=115200)
ser.timeout = 0.02


In [43]:
ser.reset_input_buffer()

ft.configure_glitcher(
    power_cycle_output = faultier.OUT_MUX0,
    power_cycle_length = 300000
)

ft.power_cycle()
d = ser.readline()
print(d)
# ser.write(b"TEST\r\n")
# d = ser.readline()
# print(d)

b'GlitchTag Bootloader\n'


In [ ]:
ft.configure_glitcher(
    trigger_source = faultier.TRIGGER_NONE,
    trigger_type = faultier.TRIGGER_NONE,
    glitch_output= faultier.OUT_CROWBAR
)

expected = b'GlitchTag Bootloader\n\x00'

import random

while True:
    delay = random.randint(610000, 615000+9000)
    pulse = random.randint(9, 14)
    # Flush potentially left-over data in serial input buffer
    ser.reset_input_buffer()

    ft.glitch(delay, pulse)

    data = ser.read(len(expected) + 100)
    if(data == b''):
        continue
    if(len(data) < len(expected)):
        continue
    if(len(data) > len(expected)+2):
        ser.timeout = 0.1
        fw = data + ser.read(0x10000)
        ser.timeout = 0.02
        f = open(f"fw_{delay}_{pulse}.bin", "wb")
        f.write(fw)
        f.close()
        print(f"{delay} {pulse} Success! Dumped {len(fw)} bytes")

Example success:
```
613676 8 Success! b'G@\x13\x00 )\x08\x00\x003F\x00\x00\x15\x08\x00\x00\x15\x08\x00\x00\x15\x08\x00\x00\x15\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xd9\t\x00\x00\x15\x08\x00\x00\x00\x00\x00\x00\xa5\t\x00\x00\x15\x08\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e' b'\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00e\x0b\x00\x00G'


610231 12 Success! Dumped 354 bytes
617065 10 Success! Dumped 43 bytes
611091 9 Success! Dumped 357 bytes
615606 12 Success! Dumped 88 bytes
616653 12 Success! Dumped 31 bytes
```

```